# Solutions · 06 · Reaction-diffusion: fronts and patterns

Worked solutions to the exercises of [notebook 06](../notebooks/06_reaction_diffusion_fronts_and_patterns.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, stiffness, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Measure the Fisher front speed on grids with dx = 0.3, 0.15, 0.075 and extrapolate the asymptotic speed at t = 15; compare with 2 and with Bramson's 2 - 3/(2t).

In [2]:
rows = []
for dx in (0.3, 0.15, 0.075):
    p = pr.fisher(n=int(round(60 / dx)))
    s = p.solve("bdf", 0.02, order=2, n_out=15)
    pos = np.array([pr.front_position(p.x, y) for y in s.y])
    speed = (pos[-1] - pos[-2]) / (s.t[-1] - s.t[-2])
    rows.append((dx, speed))
print(pd.DataFrame(rows, columns=["dx", "speed at t = 14.5"]).to_string(index=False))
print(f"Bramson at t = 14.5: 2 - 3/(2 t) = {2 - 3 / (2 * 14.5):.3f}")

   dx  speed at t = 14.5
0.300           1.890859
0.150           1.885646
0.075           1.884395
Bramson at t = 14.5: 2 - 3/(2 t) = 1.897


The measured speed hardly changes with the grid: the front is resolved already at Δx = 0.3 (its width is of
order 1). The remaining gap to 2 is the slow approach itself - Bramson's 2 − 3/(2t) gives 1.90 at t = 14.5,
which is what the computation shows. A convergence study distinguishes a discretisation error from a slow
transient; without it the two look alike.

## Exercise 2

Change Gray-Scott to F = 0.03, k = 0.062 and to F = 0.025, k = 0.05 (worms, then chaos). Does explicit RK4 remain adequate? Confirm with the spectrum at a developed pattern.

In [3]:
for F, k, label in ((0.03, 0.062, "worms"), (0.025, 0.05, "chaotic spots")):
    p = pr.gray_scott(n=48, F=F, k=k)
    s = p.solve("rk4", 1.0, t_end=2000.0, n_out=1)
    lam = stiffness.spectrum(p.rhs, s.y[-1])
    print(f"F = {F}, k = {k} ({label}): spectrum from {lam.real.min():.3f} to {lam.real.max():+.4f}; RK4 limit {stiffness.explicit_step_limit(lam, 'rk4'):.2f}; v range [{s.y[-1][48*48:].min():.3f}, {s.y[-1][48*48:].max():.3f}]")

F = 0.03, k = 0.062 (worms): spectrum from -0.421 to +0.0317; RK4 limit 6.62; v range [0.000, 0.401]


F = 0.025, k = 0.05 (chaotic spots): spectrum from -0.419 to -0.0005; RK4 limit 6.65; v range [0.216, 0.225]


For both parameter pairs the eigenvalues stay within a few units of the origin (the diffusive part, 4D/Δx²
≈ 0.2, is tiny), so RK4's step limit is of order one and h = 1 remains adequate - as long as the grid is
coarse. On a finer grid the diffusive eigenvalues grow like 1/Δx² and explicit integration eventually becomes
the costlier choice; the spectrum, recomputed, says when.

## Exercise 3

**Implement it yourself:** write the IMEX CNAB2 step (Crank-Nicolson for diffusion, second-order Adams-Bashforth for the reaction) and reproduce the library's front position.

In [4]:
p = pr.fisher()
g = p.grid
fe = p.params["rhs_explicit"]; fi = p.params["rhs_implicit"]
from engmolines import linalg as la
A = la.from_banded(la.banded_jacobian(lambda w: fi(0.0, w), p.y0, 1, 1), 1, 1)       # the linear diffusion operator
h = 0.02; u = p.y0.copy(); fe_prev = None
M = np.eye(len(u)) - 0.5 * h * A
for k in range(int(round(15 / h))):
    f_now = fe(0, u)
    if fe_prev is None:
        rhs = u + h * f_now                                   # first step: IMEX Euler (no history)
        u = np.linalg.solve(np.eye(len(u)) - h * A, rhs)
    else:
        rhs = u + h * (1.5 * f_now - 0.5 * fe_prev) + 0.5 * h * (A @ u)
        u = np.linalg.solve(M, rhs)
    fe_prev = f_now
lib = p.solve("imex", h, n_out=1).y[-1]
print(f"hand-written CNAB2 vs library: front at {pr.front_position(p.x, u):.3f} vs {pr.front_position(p.x, lib):.3f}; max difference {np.max(np.abs(u - lib)):.1e}")

hand-written CNAB2 vs library: front at 28.816 vs 28.816; max difference 6.1e-09


CNAB2 applies the trapezoidal rule to the linear diffusion (one fixed matrix I − hA/2) and second-order
Adams-Bashforth to the reaction; the first step, lacking history, is an IMEX Euler step. The hand-written
loop reproduces the library's front.